# 12 — DAG Task Scheduler (Amazon FAR style)

Run tasks with dependencies (a build system / workflow engine). Parts 1-3 build the core (concurrency
at Part 3); Parts 4-5 are stretch tasks (failure propagation, then parallel cost precompute). Fill
stubs, run each test cell, peek at the collapsed `ref_` solutions only after trying.

Dependencies are `deps: dict[task, set[prereq_tasks]]` (every task is a key, even with no prereqs).

---

## Part 1 — Ready tasks

`get_ready(deps, done) -> list`: the tasks not yet in `done` whose **every** dependency is already in
`done` (return them sorted, for determinism).

**Lock down:** a task with no deps is ready immediately; this is the primitive the scheduler calls
each time a task completes.

In [ ]:
def get_ready(deps, done):
    raise NotImplementedError

In [ ]:
# --- Part 1 tests ---
def _t1():
    deps = {"a": set(), "b": {"a"}, "c": {"a"}, "d": {"b", "c"}}
    assert get_ready(deps, set()) == ["a"]
    assert get_ready(deps, {"a"}) == ["b", "c"]
    assert get_ready(deps, {"a", "b", "c"}) == ["d"]
    assert get_ready(deps, {"a", "b", "c", "d"}) == []
    print("Part 1 OK")

_t1()

## Part 2 — Topological order

`topological_order(deps) -> list`: a linear order in which every task appears **after** all its
dependencies. Raise `ValueError` if the graph has a cycle (no valid order).

**Lock down:** Kahn's algorithm (repeatedly emit in-degree-0 nodes) or DFS post-order; a cycle is
detected when you can't emit all nodes. There can be many valid orders — produce any one.

In [ ]:
def topological_order(deps):
    raise NotImplementedError

In [ ]:
# --- Part 2 tests ---
def _t2():
    deps = {"a": set(), "b": {"a"}, "c": {"a"}, "d": {"b", "c"}}
    order = topological_order(deps)
    pos = {t: i for i, t in enumerate(order)}
    assert len(order) == 4 and set(order) == set(deps)
    for t, ps in deps.items():
        for p in ps:
            assert pos[p] < pos[t]                    # dependency comes first
    try:
        topological_order({"x": {"y"}, "y": {"x"}})
    except ValueError:
        pass
    else:
        raise AssertionError("expected ValueError on cycle")
    print("Part 2 OK")

_t2()

## Part 3 — Concurrent execution

`run_dag(deps, run_fn, num_workers) -> (order, results)`: execute tasks across `num_workers` threads,
running `run_fn(task)` for each, where a task may start **only after all its dependencies have
completed**. Independent ready tasks run in parallel. Return the completion `order` and a
`{task: result}` dict.

**The crux:** as each task finishes, re-check which tasks just became ready and dispatch them; the run
ends when every task is done. **Discuss:** scheduling on completion (not up front), the
check-then-dispatch race on the shared `done`/`scheduled` sets, and how workers know to stop (sentinels
once all tasks complete). `order` must always respect dependencies.

In [ ]:
import queue
import threading


def run_dag(deps, run_fn, num_workers):
    raise NotImplementedError

In [ ]:
# --- Part 3 tests ---
def _t3():
    deps = {"a": set(), "b": {"a"}, "c": {"a"}, "d": {"b", "c"}, "e": {"d"}}
    for _ in range(5):                                 # repeat to surface races
        order, results = run_dag(deps, lambda t: t.upper(), num_workers=4)
        pos = {t: i for i, t in enumerate(order)}
        assert set(order) == set(deps) and len(order) == 5
        for t, ps in deps.items():
            for p in ps:
                assert pos[p] < pos[t]                 # never ran before a dependency
        assert results == {t: t.upper() for t in deps}
    assert run_dag({}, lambda t: t, 3) == ([], {})     # empty DAG terminates
    print("Part 3 OK")

_t3()

## Part 4 (stretch) — Failure propagation

`run_dag_safe(deps, run_fn) -> (results, failed, skipped)`: if a task's `run_fn` raises, mark it
**failed**; any task that (transitively) depends on a failed or skipped task is **skipped**, not run;
everything else still runs. Raise `ValueError` on a cycle.

**Discuss:** fail-fast vs run-as-much-as-possible, ret/retry policy per task, and why processing in
topological order makes "did an upstream fail?" a simple check.

In [ ]:
def run_dag_safe(deps, run_fn):
    raise NotImplementedError

In [ ]:
# --- Part 4 tests ---
def _t4():
    deps = {"a": set(), "b": {"a"}, "c": {"b"}, "d": set()}

    def run_fn(t):
        if t == "b":
            raise RuntimeError("boom")
        return t

    results, failed, skipped = run_dag_safe(deps, run_fn)
    assert failed == {"b"}
    assert skipped == {"c"}                            # c depends on failed b
    assert results == {"a": "a", "d": "d"}             # a and d unaffected
    try:
        run_dag_safe({"x": {"y"}, "y": {"x"}}, lambda t: t)
    except ValueError:
        pass
    else:
        raise AssertionError("expected ValueError on cycle")
    print("Part 4 OK")

_t4()

## Part 5 (stretch) — Parallel cost precompute

`parallel_costs(items, num_procs) -> dict`: compute a CPU-bound cost for each node across processes
with `ProcessPoolExecutor`; worker is `dagsched_workers.node_cost`. (Useful for critical-path /
scheduling heuristics before you run the DAG.)

**Discuss:** node costs are independent (embarrassingly parallel); GIL (cost is CPU-bound ->
processes); and how per-node costs feed list-scheduling / critical-path priority.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import dagsched_workers


def parallel_costs(items, num_procs) -> dict:
    raise NotImplementedError

In [ ]:
# --- Part 5 tests ---
def _t5():
    items = [1, 2, 3, 10]
    assert parallel_costs(items, 2) == {n: sum(i * i for i in range(n)) for n in items}
    print("Part 5 OK")

_t5()

## Likely follow-ups
- Priority scheduling by critical path / node cost; bounded parallelism per resource.
- Retries with backoff per task; partial re-runs (only stale subgraphs).
- Distributed scheduling: idempotent tasks, at-least-once execution, leases.

---
## Reference solutions (don't peek until you've tried)

In [ ]:
import queue
import threading
from collections import deque
from concurrent.futures import ProcessPoolExecutor
import dagsched_workers


def ref_get_ready(deps, done):
    done = set(done)
    return sorted(t for t in deps if t not in done and deps[t] <= done)


def ref_topological_order(deps):
    indeg = {t: 0 for t in deps}
    adj = {t: [] for t in deps}
    for t, ps in deps.items():
        for p in ps:
            adj[p].append(t)
            indeg[t] += 1
    q = deque(sorted(t for t in deps if indeg[t] == 0))
    order = []
    while q:
        u = q.popleft()
        order.append(u)
        for v in sorted(adj[u]):
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    if len(order) != len(deps):
        raise ValueError("cycle")
    return order


def ref_run_dag(deps, run_fn, num_workers):
    done, scheduled, results, order = set(), set(), {}, []
    lock = threading.Lock()
    ready = queue.Queue()
    total = len(deps)

    def schedule_ready():                              # call under lock
        for t in deps:
            if t not in scheduled and deps[t] <= done:
                scheduled.add(t)
                ready.put(t)

    with lock:
        schedule_ready()
        if len(done) == total:                         # empty DAG: nothing to do
            for _ in range(num_workers):
                ready.put(None)

    def worker():
        while True:
            t = ready.get()
            if t is None:
                return
            r = run_fn(t)
            with lock:
                results[t] = r
                order.append(t)
                done.add(t)
                schedule_ready()
                all_done = len(done) == total
            if all_done:
                for _ in range(num_workers):
                    ready.put(None)

    threads = [threading.Thread(target=worker) for _ in range(num_workers)]
    for th in threads:
        th.start()
    for th in threads:
        th.join()
    return order, results


def ref_run_dag_safe(deps, run_fn):
    order = ref_topological_order(deps)                # raises on cycle
    results, failed, skipped = {}, set(), set()
    for t in order:
        if any(p in failed or p in skipped for p in deps[t]):
            skipped.add(t)
            continue
        try:
            results[t] = run_fn(t)
        except Exception:                              # noqa: BLE001
            failed.add(t)
    return results, failed, skipped


def ref_parallel_costs(items, num_procs):
    with ProcessPoolExecutor(max_workers=num_procs) as ex:
        return dict(zip(items, ex.map(dagsched_workers.node_cost, items)))


_d = {"a": set(), "b": {"a"}, "c": {"a", "b"}}
assert ref_get_ready(_d, {"a"}) == ["b"]
_o = ref_topological_order(_d); assert _o.index("a") < _o.index("b") < _o.index("c")
_order, _res = ref_run_dag(_d, lambda t: t, 3)
assert _res == {"a": "a", "b": "b", "c": "c"} and _order.index("a") < _order.index("c")
_r, _f, _s = ref_run_dag_safe({"a": set(), "b": {"a"}}, lambda t: (_ for _ in ()).throw(ValueError()) if t == "a" else t)
assert _f == {"a"} and _s == {"b"}
assert ref_parallel_costs([1, 4], 2) == {1: 0, 4: 14}
print("reference solutions OK")